### Imports/installs

In [66]:
import re
import numpy as np
import pandas as pd, torch, gc
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score, classification_report, precision_recall_curve

%pip install -q transformers torch tqdm
from tqdm.auto import tqdm
from transformers import pipeline
from detoxify import Detoxify


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Relabel data

In [81]:
# input and output files
INPUT_CSV  = "twitter_parsed_dataset.csv"
OUTPUT_CSV = "twitter_parsed_dataset_relabeled.csv"

# match columns
TEXT_COL   = "Text"
BULLY_COL  = "oh_label"
ANN_COL    = "Annotation"

# set up model and device
TRUNC       = 256
BATCH_SIZE  = 32 if torch.cuda.is_available() else 16
MODEL_NAME  = "typeform/distilbert-base-uncased-mnli"
DEVICE      = 0 if torch.cuda.is_available() else -1
CATS        = ["racism","sexism","insult"]

# threshold for flipping to insult
INSULT_THRESH = 0.6
MARGIN        = 0.10

# csv to datagrame
df = pd.read_csv(INPUT_CSV)
df[BULLY_COL] = pd.to_numeric(df[BULLY_COL], errors="coerce").fillna(0).astype(int).clip(0,1)

df[ANN_COL] = np.where(df[BULLY_COL]==0, "none", df.get(ANN_COL, "sexism"))

idxs  = df.index[df[BULLY_COL]==1].tolist()
texts = df.loc[idxs, TEXT_COL].astype(str).str.slice(0, TRUNC).tolist()
N = len(texts)

if N > 0:
    zs = pipeline("zero-shot-classification", model=MODEL_NAME, device=DEVICE)

    S = np.zeros((N, len(CATS)), dtype=np.float32)
    pbar = tqdm(total=N, desc="Zero-shot on bullying only", unit="rows")
    for i in range(0, N, BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        out = zs(batch, candidate_labels=CATS, multi_label=False)
        for j, o in enumerate(out):
            m = dict(zip(o["labels"], o["scores"]))
            S[i+j, 0] = m.get("racism", 0.0)
            S[i+j, 1] = m.get("sexism", 0.0)
            S[i+j, 2] = m.get("insult", 0.0)
        pbar.update(len(batch))
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    pbar.close()

    for pos, row_idx in enumerate(idxs):
        cur = str(df.at[row_idx, ANN_COL]).strip().lower()
        if cur not in ("racism","sexism"):
            continue

        cur_idx = 0 if cur == "racism" else 1
        cur_prob = S[pos, cur_idx]
        ins_prob = S[pos, 2]

        if (ins_prob >= INSULT_THRESH) and (ins_prob >= cur_prob + MARGIN):
            df.at[row_idx, ANN_COL] = "insult"
        else:
            pass

df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)
print("All rows:\n", df[ANN_COL].value_counts())
print("Bullying only:\n", df.loc[df[BULLY_COL]==1, ANN_COL].value_counts())

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
Device set to use cpu


Zero-shot on bullying only:   0%|          | 0/5347 [00:00<?, ?rows/s]

Saved: twitter_parsed_dataset_relabeled.csv
All rows:
 Annotation
none      11504
sexism     2693
racism     1712
insult      942
Name: count, dtype: int64
Bullying only:
 Annotation
sexism    2693
racism    1712
insult     942
Name: count, dtype: int64


### Remove usernames/PII

In [83]:
USERNAME_PATTERNS = [r"@\w+", r"@\[[^\]]+\]", r"@\(.*?\)", r"@[^ \t\n\r\f\v]+"]
def remove_usernames(t):
    if not isinstance(t, str):
        return ""
    for pat in USERNAME_PATTERNS:
        t = re.sub(pat, " ", t)
    return re.sub(r"\s+", " ", t).strip()

### Change labels: text, bully, category

In [84]:
def normalize_labels_for_csv(df):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    for col in ["Text","oh_label","Annotation"]:
        if col not in df.columns:
            raise ValueError(f"Missing column {col}. Found {df.columns.tolist()}")

    df["text"] = df["Text"].astype(str).map(remove_usernames)

    bully = pd.to_numeric(df["oh_label"], errors="coerce").fillna(0).astype(int).clip(0,1)
    df["bully"] = bully

    cat = df["Annotation"].astype(str).str.lower().str.strip()
    cat = np.where(bully == 0, "none", cat)
    cat = pd.Series(cat).where(pd.Series(cat).isin(["none","racism","sexism","insult"]), other="none")
    df["category"] = cat

    return df[["text","bully","category"]]

### Training: train/val/test split, build pipelines, tune threshold

In [85]:
def split_sets(df, seed=42):
    train_val, test = train_test_split(df, test_size=0.15, random_state=seed, stratify=df["bully"])
    train, val = train_test_split(train_val, test_size=0.15/(1-0.15), random_state=seed, stratify=train_val["bully"])
    return train, val, test

In [86]:
def build_bin_pipeline():
    word_vec = TfidfVectorizer(
        ngram_range=(1,2),
        min_df=5,
        max_df=0.95,
        max_features=50000,
        sublinear_tf=True,
        strip_accents="unicode",
        lowercase=True,
    )
    char_vec = TfidfVectorizer(
        analyzer="char",
        ngram_range=(3,4),
        min_df=5,
        max_df=0.95,
        max_features=30000,
        sublinear_tf=True,
        lowercase=True,
    )

    col = ColumnTransformer(
        [("word", word_vec, "text"),
         ("char", char_vec, "text")],
        remainder="drop",
        sparse_threshold=1.0,
    )

    return Pipeline([
        ("prep", col),
        ("clf", LogisticRegression(
            C=0.3,
            class_weight="balanced",
            max_iter=4000,
            solver="liblinear",
            penalty="l2",
        ))
    ])

In [87]:
def build_cat_pipeline():
    word_vec = TfidfVectorizer(
        ngram_range=(1,2),
        min_df=10,
        max_df=0.90,
        max_features=30000,
        sublinear_tf=True,
        strip_accents="unicode",
        lowercase=True,
        stop_words="english",
    )
    char_vec = TfidfVectorizer(
        analyzer="char",
        ngram_range=(3,4),
        min_df=10,
        max_df=0.90,
        max_features=20000,
        sublinear_tf=True,
        lowercase=True,
    )

    col = ColumnTransformer(
        [("word", word_vec, "text"),
         ("char", char_vec, "text")],
        remainder="drop",
        sparse_threshold=1.0,
    )

    feat_sel = SelectKBest(chi2, k=40000)

    svc = LinearSVC(C=0.1, class_weight="balanced")
    cat_clf = CalibratedClassifierCV(svc, method="sigmoid", cv=3, n_jobs=-1)

    return Pipeline([
        ("prep", col),
        ("select", feat_sel),
        ("clf", cat_clf),
    ])

In [88]:
def tune_threshold(bin_model, X_val, y_val):
    proba_pos = bin_model.predict_proba(X_val)[:,1]
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.2, 0.8, 121):
        pred = (proba_pos >= t).astype(int)
        f1 = f1_score(y_val, pred, pos_label=1)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

### Model evaluation

In [89]:
def eval_report(model, X, y, title):
    p = model.predict(X)
    print(f"\n== {title} ==")
    print("Accuracy:", f"{accuracy_score(y,p):.4f}")
    print("F1 (macro):", f"{f1_score(y,p,average='macro'):.4f}")
    print(classification_report(y,p,digits=4))

In [90]:
def eval_binary(model, X, y, thr, title):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= thr).astype(int)
    acc = accuracy_score(y, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y, preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(y, probs)
    except ValueError:
        auc = float("nan")
    print(f"\n== {title} ==")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}  ROC-AUC: {auc:.4f}")

### Model Prediction

In [91]:
def _proba_from_clf(clf, X):
    if hasattr(clf, "predict_proba"):
        return clf.predict_proba(X)
    if hasattr(clf, "decision_function"):
        s = clf.decision_function(X)
        s = np.atleast_2d(s)
        m = np.max(s, axis=1, keepdims=True)
        exp = np.exp(s - m)
        return exp / np.sum(exp, axis=1, keepdims=True)
    raise AttributeError("Classifier has neither predict_proba nor decision_function.")

In [92]:
def _count_hits(text, cues):
    t = text.lower()
    return sum(1 for c in cues if c in t)

In [93]:
def _cue_scores(text, cue_dict, scale=3.0):
    # count -> [0,1] via tanh; per class
    scores = {}
    for lab, cues in cue_dict.items():
        c = _count_hits(text, cues)
        scores[lab] = np.tanh(c / scale)  # gentle saturation
    return scores  # e.g. {"racism": 0.0..1.0, "sexism": ..., "insult": ...}

In [94]:
def predict_pipeline(texts, threshold=None, backstop=True, cat_conf=0.70):
    X_df = pd.Series(texts, name="text").to_frame()

    if threshold is None:
        threshold = max(0.35, t_star - 0.10)

    # binary
    probs = bin_model.predict_proba(X_df)[:, 1]
    bpred = (probs >= threshold).astype(int)
    cats  = np.array(["none"] * len(texts), dtype=object)

    # no category model available: assign a default for bullying rows
    if cat_model is None:
        cats[bpred == 1] = "insult"
        return [(txt, int(b), str(c)) for txt, b, c in zip(texts, bpred, cats)]

    # category model
    prep = cat_model.named_steps.get("prep")
    clf  = cat_model.named_steps.get("clf")
    Xc   = prep.transform(X_df)
    cprobs = _proba_from_clf(clf, Xc)          # [n, K]
    classes = np.asarray(clf.classes_)         # ensure ndarray

    valid = ("racism","sexism","insult")
    # optional backstop: if category is confident, flip binary to 1
    if backstop:
        argmax_idx = np.argmax(cprobs, axis=1)
        chat = classes[argmax_idx]
        cmax = np.max(cprobs, axis=1)
        for i in range(len(texts)):
            if bpred[i] == 0 and cmax[i] >= cat_conf and chat[i] in valid:
                bpred[i] = 1
                cats[i]  = chat[i]

    # assign categories for bullying rows via pure model probs
    # build a per-row dict to handle missing classes
    for i in range(len(texts)):
        if bpred[i] != 1:
            continue
        # map available class -> prob for this row
        row_probs = {cls: float(cprobs[i, j]) for j, cls in enumerate(classes) if cls in valid}
        if row_probs:
            cats[i] = max(row_probs.keys(), key=lambda k: row_probs[k])
        else:
            cats[i] = "insult"

    # any remaining bullying rows without a label: use direct predict
    fill_idx = np.where((bpred == 1) & (cats == "none"))[0]
    if len(fill_idx):
        cats[fill_idx] = cat_model.predict(X_df.iloc[fill_idx])

    return [(txt, int(b), str(c)) for txt, b, c in zip(texts, bpred, cats)]

### Run: clean relabeled data, train/evaluate model

In [95]:
# --- load & normalize ---
df_raw = pd.read_csv("twitter_parsed_dataset_relabeled.csv")
df = normalize_labels_for_csv(df_raw)
print("Bullying counts:\n", df["bully"].value_counts())
print("Bullying categories:\n", df.query("bully==1")["category"].value_counts())

# splits
train, val, test = split_sets(df, seed=42)

# --- 1) Binary: fit whole pipeline ---
bin_model = build_bin_pipeline()

with tqdm(total=1, desc="Training — Binary", leave=True) as pbar:
    bin_model.fit(train[["text"]], train["bully"])
    pbar.update(1)

# tune threshold on validation using the fitted pipeline
t_star, f1_star = tune_threshold(bin_model, val[["text"]], val["bully"])
print("Chosen threshold:", t_star, "F1@t* (val):", f1_star)

# binary metrics
eval_binary(bin_model, train[["text"]], train["bully"], t_star, "Binary — Train")
eval_binary(bin_model, val[["text"]],   val["bully"],   t_star, "Binary — Validation")
eval_binary(bin_model, test[["text"]],  test["bully"],  t_star, "Binary — Test")

# --- 2) Category (insult/racism/sexism): fit whole pipeline ---
train_b = train[train["bully"] == 1].query("category in ['insult','racism','sexism']")
val_b   = val[val["bully"] == 1].query("category in ['insult','racism','sexism']")
test_b  = test[test["bully"] == 1].query("category in ['insult','racism','sexism']")

cat_model = None
if len(train_b) and len(val_b) and len(test_b) and train_b["category"].nunique() >= 2:
    cat_model = build_cat_pipeline()

    with tqdm(total=1, desc="Training — Category", leave=True) as pbar:
        cat_model.fit(train_b[["text"]], train_b["category"])
        pbar.update(1)

    # category metrics
    eval_report(cat_model, train_b[["text"]], train_b["category"], "Category — Train")
    eval_report(cat_model, val_b[["text"]],   val_b["category"],   "Category — Validation")
    eval_report(cat_model, test_b[["text"]],  test_b["category"],  "Category — Test")
else:
    print("Not enough bullying rows (need ≥2 categories) to train a category model.")

Bullying counts:
 bully
0    11504
1     5347
Name: count, dtype: int64
Bullying categories:
 category
sexism    2693
racism    1712
insult     942
Name: count, dtype: int64


Training — Binary:   0%|          | 0/1 [00:00<?, ?it/s]

Chosen threshold: 0.48500000000000004 F1@t* (val): 0.7454764776839565

== Binary — Train ==
Accuracy: 0.8788
Precision: 0.7832  Recall: 0.8549  F1: 0.8175  ROC-AUC: 0.9454

== Binary — Validation ==
Accuracy: 0.8331
Precision: 0.7220  Recall: 0.7706  F1: 0.7455  ROC-AUC: 0.9015

== Binary — Test ==
Accuracy: 0.8358
Precision: 0.7253  Recall: 0.7768  F1: 0.7502  ROC-AUC: 0.8903


Training — Category:   0%|          | 0/1 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:783: UserWarning: k=40000 is greater than n_features=9491. All the features will be returned.
  warnings.warn(



== Category — Train ==
Accuracy: 0.8627
F1 (macro): 0.7816
              precision    recall  f1-score   support

      insult     0.8905    0.3636    0.5164       671
      racism     0.8910    0.9529    0.9209      1210
      sexism     0.8423    0.9839    0.9076      1862

    accuracy                         0.8627      3743
   macro avg     0.8746    0.7668    0.7816      3743
weighted avg     0.8667    0.8627    0.8418      3743


== Category — Validation ==
Accuracy: 0.8317
F1 (macro): 0.7014
              precision    recall  f1-score   support

      insult     0.6500    0.2114    0.3190       123
      racism     0.8824    0.9266    0.9040       259
      sexism     0.8184    0.9548    0.8813       420

    accuracy                         0.8317       802
   macro avg     0.7836    0.6976    0.7014       802
weighted avg     0.8132    0.8317    0.8024       802


== Category — Test ==
Accuracy: 0.8042
F1 (macro): 0.6803
              precision    recall  f1-score   support


In [96]:
# --- 3) End‑to‑end predict ---
sample = ["@user you are so dumb",
          "Have a nice day",
          "women don’t belong here",
          "that policy is terrible"]
result = predict_pipeline(sample)
result

[('@user you are so dumb', 1, 'insult'),
 ('Have a nice day', 0, 'none'),
 ('women don’t belong here', 1, 'sexism'),
 ('that policy is terrible', 0, 'none')]